# Forced Directional Handover Validation

This notebook tests the full HRL handover chain without PPO training.

Goal:

```text
forced directional bias
→ lower/executor offset
→ directional admission budget
→ sequential A3 handover
→ demand PRB moves from source to target
→ paper_cost reward observes the change
```

Expected behavior:

```text
gNB1 eMBB is overloaded
force gNB1 → gNB0 eMBB = -1
force gNB1 → gNB2 eMBB = -1
max_handovers_per_local_step = 1
handover should occur sequentially, not simultaneously
```

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Change this if the notebook is not launched from the project root.
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from global_ppo_3gnb_env import GlobalPPO3GNBEnv, SLICE_TYPES

print("Project root:", PROJECT_ROOT)
print("Slice types:", SLICE_TYPES)

## Helper functions

`make_directional_action()` builds the flattened PPO action tensor using the environment's own neighbor ordering.

In [ ]:
def make_directional_action(env, entries):
    # Build flattened directional action tensor.
    # entries: dict[(source_gnb, target_gnb, slice_type)] = bias_value
    action_tensor = np.zeros(
        (env.n_gnbs, env.max_neighbors, len(env.slice_types)),
        dtype=np.float32,
    )

    for (source, target, slice_type), value in entries.items():
        source = int(source)
        target = int(target)
        neighbors = tuple(env.neighbors[source])
        if target not in neighbors:
            raise ValueError(f"gNB{target} is not a neighbor of gNB{source}. Neighbors: {neighbors}")

        neighbor_slot = neighbors.index(target)
        slice_idx = env.slice_types.index(slice_type)
        action_tensor[source, neighbor_slot, slice_idx] = float(value)

    return action_tensor.reshape(-1), action_tensor


def as_array(value):
    if isinstance(value, str):
        return np.asarray(json.loads(value), dtype=float)
    return np.asarray(value, dtype=float)


def maybe_json(value):
    if isinstance(value, str):
        try:
            return json.loads(value)
        except Exception:
            return value
    return value


def matrix_df(matrix, env):
    matrix = np.asarray(matrix, dtype=float)
    return pd.DataFrame(
        matrix,
        index=[f"gNB{idx}" for idx in range(matrix.shape[0])],
        columns=list(env.slice_types),
    )

## Create the environment

This uses the controllable scenario because the UEs are intentionally placed where directional A3 offset should be able to move them.

Important settings:
- `upper_reward_mode="paper_cost"`
- `safe_admission_enabled=True`
- `max_handovers_per_local_step=1`
- short residence/cooldown guards for testing

In [ ]:
env = GlobalPPO3GNBEnv(
    seed=7,
    training_scenarios="jain_balance_controllable",
    scenario_selection="cycle",
    scenario_mode="curriculum",

    # Timing
    upper_window_seconds=1.0,
    local_steps_per_global=12,
    radio_substeps=100,

    # Sequential handover
    max_handovers_per_local_step=1,
    max_handovers_per_episode=20,
    max_handovers_per_ue_episode=1,

    # Make test easier / less blocked by mobility guards
    warmup_steps=0,
    post_handover_settle_steps=4,
    a3_handover_cooldown_s=0.0,
    a3_min_residence_s=0.0,
    handover_pingpong_guard_s=0.0,
    a3_pingpong_threshold_s=1.0,

    # Intended current setup
    safe_admission_enabled=True,
    upper_reward_mode="paper_cost",
    terminal_reward_only=False,
)

obs, reset_info = env.reset()

print("upper_reward_mode:", env.upper_reward_mode)
print("safe_admission_enabled:", env.safe_admission_enabled)
print("max_handovers_per_local_step:", env.max_handovers_per_local_step)
print("base_env.max_handovers_per_step:", env.base_env.max_handovers_per_step)
print("local_step_seconds:", env.local_step_seconds)
print("radio_tick_seconds:", env.radio_tick_seconds)
print("radio_substeps:", env.radio_substeps)
print("handover_ttt:", env.base_env.handover_ttt)
print("effective_ttt_s:", env.base_env.handover_ttt * env.local_step_seconds)

## Check initial demand load

We expect eMBB demand to be concentrated on gNB1.

In [ ]:
start_demand_load = np.asarray(env._load_matrix(), dtype=float)

display(matrix_df(start_demand_load, env))

embb_idx = env.slice_types.index("eMBB")
print("Initial eMBB demand load per gNB:", start_demand_load[:, embb_idx])
assert start_demand_load[1, embb_idx] > start_demand_load[0, embb_idx]
assert start_demand_load[1, embb_idx] > start_demand_load[2, embb_idx]

## Force the directional action

We force:

```text
gNB1 → gNB0 eMBB = -1.0
gNB1 → gNB2 eMBB = -1.0
all other directions/slices = 0
```

In [ ]:
action, action_tensor = make_directional_action(
    env,
    {
        (1, 0, "eMBB"): -1.0,
        (1, 2, "eMBB"): -1.0,
    },
)

print("Action tensor shape:", action_tensor.shape)
print("Neighbor order:", env.neighbors)
print("Action tensor [source, neighbor_slot, slice]:")
print(action_tensor)

## Run one upper step

This triggers the full chain:
- action → directional bias
- lower executor → A3 offsets
- admission budget
- sequential local steps
- reward computation

In [ ]:
obs, reward, terminated, truncated, info = env.step(action)

print("Reward:", reward)
print("Terminated:", terminated)
print("Truncated:", truncated)
print("upper_reward_mode:", info.get("upper_reward_mode"))
print("paper_load_source:", info.get("paper_load_source"))
print("handover_count:", info.get("handover_count"))
print("required_handover_settle_steps:", info.get("required_handover_settle_steps"))
print("effective_handover_settle_steps:", info.get("effective_handover_settle_steps"))
print("handover_settle_truncated:", info.get("handover_settle_truncated"))
print("radio_measurement_steps:", info.get("radio_measurement_steps"))

## Inspect offsets, budget, and admission use

If handover count is zero, this section tells you whether the issue is:
- offset was zeroed by the executor
- budget was zero
- admission rejected candidates
- A3 did not trigger

In [ ]:
offset_tensor = as_array(info.get("directional_offset_tensor"))
print("Directional offset tensor shape:", offset_tensor.shape)
print(offset_tensor)

print("\nAdmission capacities / quota:")
print(maybe_json(info.get("safe_admission_capacities")))

print("\nAdmission accepted / used:")
print(maybe_json(info.get("safe_admission_accepted")))

print("\nAdmission remaining:")
print(maybe_json(info.get("safe_admission_remaining")))

print("\nAdmission stats:")
print(maybe_json(info.get("safe_admission_stats")))

## Inspect handover sequence

The key test: **at most one handover per local step**.

In [ ]:
sequence = maybe_json(info.get("upper_window_handover_sequence", []))
print("Raw sequence:")
print(json.dumps(sequence, indent=2))

by_local_step = {}
for event in sequence:
    local_step = int(event.get("local_step_in_upper_window", event.get("local_step", -1)))
    by_local_step[local_step] = by_local_step.get(local_step, 0) + 1

print("Handover count by local step:", by_local_step)

if by_local_step:
    assert max(by_local_step.values()) <= 1, "More than one handover happened in one local step"
    assert min(by_local_step) >= int(env.base_env.handover_ttt), "Handover happened before A3 TTT"
else:
    print("No handovers happened. Check offsets/budget/A3 eligibility above.")

## Compare demand PRB before and after

We expect:
- gNB1 eMBB demand decreases
- gNB0 or gNB2 eMBB demand increases
- demand-load standard deviation decreases

In [ ]:
demand_start = as_array(info["demand_prb_matrix_start"])
demand_end = as_array(info["demand_prb_matrix_end"])

start_df = matrix_df(demand_start, env)
end_df = matrix_df(demand_end, env)
delta_df = end_df - start_df

print("Demand PRB start")
display(start_df)

print("Demand PRB end")
display(end_df)

print("Demand PRB delta")
display(delta_df)

start_std = float(info["gnb_total_demand_load_std_start"])
end_std = float(info["gnb_total_demand_load_std_end"])

print("Demand load std start:", start_std)
print("Demand load std end:", end_std)
print("Improvement:", start_std - end_std)

assert demand_end[1, embb_idx] < demand_start[1, embb_idx], (
    "Source gNB1 eMBB demand did not decrease",
    demand_start[:, embb_idx],
    demand_end[:, embb_idx],
    sequence,
)
assert (
    demand_end[0, embb_idx] > demand_start[0, embb_idx]
    or demand_end[2, embb_idx] > demand_start[2, embb_idx]
), (
    "Target demand did not increase",
    demand_start[:, embb_idx],
    demand_end[:, embb_idx],
    sequence,
)
assert end_std < start_std, "Demand-load std did not improve"
assert info["upper_reward_mode"] == "paper_cost"
assert info["paper_load_source"] == "demand_prbs"

## Plot eMBB demand before/after

In [ ]:
labels = [f"gNB{i}" for i in range(env.n_gnbs)]
x = np.arange(len(labels))
width = 0.35

plt.figure(figsize=(8, 4))
plt.bar(x - width / 2, demand_start[:, embb_idx], width, label="Start")
plt.bar(x + width / 2, demand_end[:, embb_idx], width, label="End")
plt.xticks(x, labels)
plt.ylabel("Demand PRB")
plt.title("eMBB demand PRB before vs after forced directional handover")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.show()

## Final verdict cell

If all assertions pass, the forced handover path is valid enough for smoke PPO training.

In [ ]:
passed = True

checks = {
    "paper_cost_active": info.get("upper_reward_mode") == "paper_cost",
    "reward_demand_based": info.get("paper_load_source") == "demand_prbs",
    "handover_happened": int(info.get("handover_count", 0)) > 0,
    "sequential_handover": (max(by_local_step.values()) <= 1) if by_local_step else False,
    "source_demand_decreased": demand_end[1, embb_idx] < demand_start[1, embb_idx],
    "target_demand_increased": (
        demand_end[0, embb_idx] > demand_start[0, embb_idx]
        or demand_end[2, embb_idx] > demand_start[2, embb_idx]
    ),
    "demand_std_improved": end_std < start_std,
}

for name, ok in checks.items():
    print(f"{name:30s}: {'PASS' if ok else 'FAIL'}")
    passed = passed and bool(ok)

print("\nFINAL:", "PASS - forced directional handover works" if passed else "FAIL - inspect failed checks above")

env.close()